# Load NMDC taxonomy summary files (issue #77)

Downloads the NOW-tier taxonomy summary files from `data.microbiomedata.org` via HTTP,
parses each format, and writes one Parquet file per `data_object_type`.

Resolves issue #57 for small files: direct HTTP download is sufficient (no Globus needed).

| `data_object_type` | ~files | ~total | Format |
|---|---|---|---|
| `GOTTCHA2 Classification Report` | 4,243 | 0.1 GB | headered TSV |
| `Kraken2 Classification Report` | 4,243 | 2.1 GB | 6-col no-header TSV |
| `GTDBTK Bacterial Summary` | 3,535 | ~0 GB | headered TSV, one row per MAG bin |
| `GTDBTK Archaeal Summary` | 3,535 | ~0 GB | headered TSV, one row per MAG bin |
| `CheckM Statistics` | 3,535 | ~0 GB | fixed-width (custom parser) |

Each output Parquet includes a `workflow_run_id` column (= `was_generated_by` from `data_object_set`).
Biosample join requires a subsequent join via `workflow_execution_set.was_informed_by`.

In [ ]:
import io
import os
import re
import time
from pathlib import Path

import pandas as pd
import requests

KBASE_TOKEN = os.environ["KBASE_TOKEN"]
KBASE_MCP_URL = os.environ.get("KBASE_MCP_URL", "https://hub.berdl.kbase.us/apis/mcp")
OUT_DIR = Path(os.environ.get("TAXONOMY_OUT_DIR", "loaded_taxonomy"))
OUT_DIR.mkdir(exist_ok=True)

# Set to True to stop after the first file of each type (for testing)
DRY_RUN = False

In [ ]:
# ── BERDL query helpers ───────────────────────────────────────────────────────

def _query_page(sql: str, limit: int = 1000) -> list[dict]:
    r = requests.post(
        f"{KBASE_MCP_URL}/delta/tables/query",
        headers={"Authorization": f"Bearer {KBASE_TOKEN}", "Content-Type": "application/json"},
        json={"query": sql, "limit": limit},
        timeout=60,
    )
    r.raise_for_status()
    payload = r.json()
    # Delta REST API returns {"result": [...]} where items are dicts or lists.
    # Normalise to list-of-dicts regardless.
    rows = payload.get("result", payload) if isinstance(payload, dict) else payload
    if not rows:
        return []
    if isinstance(rows[0], dict):
        return rows
    # columnar: payload has "schema" + "data" arrays
    cols = [f["name"] for f in payload["schema"]["fields"]]
    return [dict(zip(cols, row)) for row in rows]


def query_all(base_sql: str, page_size: int = 1000) -> pd.DataFrame:
    """Page through a BERDL query using LIMIT/OFFSET and return a DataFrame."""
    all_rows: list[dict] = []
    offset = 0
    while True:
        paged = f"{base_sql} LIMIT {page_size} OFFSET {offset}"
        batch = _query_page(paged, limit=page_size)
        all_rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    return pd.DataFrame(all_rows)

In [ ]:
# ── Fetch the URL manifest from data_object_set ───────────────────────────────

TARGET_TYPES = [
    "GOTTCHA2 Classification Report",
    "Kraken2 Classification Report",
    "GTDBTK Bacterial Summary",
    "GTDBTK Archaeal Summary",
    "CheckM Statistics",
]

type_list = ", ".join(f"'{t}'" for t in TARGET_TYPES)

manifest_sql = f"""
SELECT id, url, was_generated_by, file_size_bytes, md5_checksum
FROM nmdc_metadata.data_object_set
WHERE data_object_type IN ({type_list})
  AND url IS NOT NULL
  AND url LIKE 'https://data.microbiomedata.org/%'
ORDER BY data_object_type, id
"""

manifest = query_all(manifest_sql)
# Re-attach data_object_type so we can group by it
# (not in SELECT above — add it)
manifest_sql_with_type = f"""
SELECT id, url, data_object_type, was_generated_by, file_size_bytes, md5_checksum
FROM nmdc_metadata.data_object_set
WHERE data_object_type IN ({type_list})
  AND url IS NOT NULL
  AND url LIKE 'https://data.microbiomedata.org/%'
ORDER BY data_object_type, id
"""
manifest = query_all(manifest_sql_with_type)
print(f"{len(manifest):,} data objects found")
print(manifest.groupby("data_object_type").size().to_string())

In [ ]:
# ── Format parsers ────────────────────────────────────────────────────────────

# GOTTCHA2: headered TSV
# LEVEL  NAME  TAXID  READ_COUNT  TOTAL_BP_MAPPED  TOTAL_BP_MISMATCH
#        LINEAR_LEN  LINEAR_DOC  ROLLUP_DOC  REL_ABUNDANCE
def parse_gottcha2(text: str) -> pd.DataFrame:
    return pd.read_csv(io.StringIO(text), sep="\t")


# Kraken2: no header, 6 cols
# pct_clade  clade_reads  direct_reads  rank  taxid  name
_KRAKEN2_COLS = ["pct_clade", "clade_reads", "direct_reads", "rank", "taxid", "name"]
def parse_kraken2(text: str) -> pd.DataFrame:
    return pd.read_csv(io.StringIO(text), sep="\t", header=None, names=_KRAKEN2_COLS)


# GTDB-tk: headered TSV with a subset of columns we care about
# user_genome  classification  fastani_ani  fastani_af  ...
def parse_gtdbtk(text: str) -> pd.DataFrame:
    df = pd.read_csv(io.StringIO(text), sep="\t")
    # Normalise: some files use 'user_genome', others 'name'
    if "user_genome" in df.columns and "name" not in df.columns:
        df = df.rename(columns={"user_genome": "name"})
    return df


# CheckM: fixed-width report, not a clean TSV despite the .tsv extension.
# Format:
#   Bin Id  Marker lineage  # genomes  # markers  # marker sets
#   0  1  2  3  4  5  Completeness  Contamination  Strain heterogeneity
# The header has a dashed separator line; data rows follow.
_CHECKM_DATA_RE = re.compile(
    r'^\s*(\S+)\s+(\S.*?)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+'
    r'([\d.]+)\s+([\d.]+)\s+([\d.]+)\s*$'
)
_CHECKM_COLS = [
    "bin_id", "marker_lineage", "n_genomes", "n_markers", "n_marker_sets",
    "n_0", "n_1", "n_2", "n_3", "n_4", "n_5",
    "completeness", "contamination", "strain_heterogeneity",
]
def parse_checkm(text: str) -> pd.DataFrame:
    rows = []
    for line in text.splitlines():
        m = _CHECKM_DATA_RE.match(line)
        if m:
            rows.append(m.groups())
    if not rows:
        return pd.DataFrame(columns=_CHECKM_COLS)
    df = pd.DataFrame(rows, columns=_CHECKM_COLS)
    for col in _CHECKM_COLS[2:]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


_PARSERS = {
    "GOTTCHA2 Classification Report": parse_gottcha2,
    "Kraken2 Classification Report": parse_kraken2,
    "GTDBTK Bacterial Summary": parse_gtdbtk,
    "GTDBTK Archaeal Summary": parse_gtdbtk,
    "CheckM Statistics": parse_checkm,
}

In [ ]:
# ── Download + parse loop ─────────────────────────────────────────────────────

def fetch_text(url: str, timeout: int = 60) -> str:
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.text


def load_type(type_name: str, rows: pd.DataFrame) -> pd.DataFrame | None:
    parser = _PARSERS[type_name]
    frames: list[pd.DataFrame] = []
    errors: list[tuple[str, str]] = []
    n = len(rows)
    if DRY_RUN:
        rows = rows.head(1)
        print(f"  DRY_RUN: processing 1 of {n} files")

    for i, row in enumerate(rows.itertuples(), 1):
        if i % 500 == 0 or i == 1:
            print(f"  {i}/{len(rows)}…", end="", flush=True)
        try:
            text = fetch_text(row.url)
            df = parser(text)
            df["workflow_run_id"] = row.was_generated_by
            df["data_object_id"] = row.id
            frames.append(df)
        except Exception as exc:
            errors.append((row.url, str(exc)))

    print()  # newline after progress
    if errors:
        print(f"  {len(errors)} errors:")
        for url, msg in errors[:5]:
            print(f"    {url}: {msg}")
        if len(errors) > 5:
            print(f"    … and {len(errors) - 5} more")

    if not frames:
        return None
    combined = pd.concat(frames, ignore_index=True)
    print(f"  {len(combined):,} rows from {len(frames):,} files")
    return combined

In [ ]:
# ── Main: one Parquet per data_object_type ────────────────────────────────────

# Safe filename: replace spaces and slashes
def _safe(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


summary: dict[str, int] = {}
t0 = time.monotonic()

for type_name, group in manifest.groupby("data_object_type"):
    out_path = OUT_DIR / f"{_safe(type_name)}.parquet"
    print(f"\n{'─'*60}")
    print(f"{type_name} ({len(group):,} files → {out_path.name})")

    combined = load_type(type_name, group)
    if combined is None:
        print("  skipped (all errors)")
        continue

    combined.to_parquet(out_path, index=False)
    summary[type_name] = len(combined)
    print(f"  wrote {out_path}")

elapsed = time.monotonic() - t0
print(f"\n{'='*60}")
print(f"Done in {elapsed/60:.1f} min")
for t, n in summary.items():
    print(f"  {t}: {n:,} rows")

In [ ]:
# ── Spot-check: GOTTCHA2 ──────────────────────────────────────────────────────

import pyarrow.parquet as pq

g2_path = OUT_DIR / "gottcha2_classification_report.parquet"
if g2_path.exists():
    g2 = pd.read_parquet(g2_path)
    print(g2.shape)
    print(g2.dtypes)
    print(g2.head(3).to_string())
    print(f"\nDistinct workflow runs: {g2['workflow_run_id'].nunique():,}")
    print(f"Distinct taxa (NAME): {g2['NAME'].nunique():,}")

## Next steps

1. **Upload to BERDL object storage** — use the BERDL upload utilities or `aws s3 cp` to push the Parquet files to the nmdc lakehouse bucket.
2. **Add biosample join** — join `workflow_run_id` to `nmdc_metadata.metagenome_annotation_activity_set.was_informed_by` → omics processing run → `has_input` → biosample ID.
3. **Register tables in BERDL catalog** — so the files are queryable via `nmdc_metadata.<type>` SQL.
4. **Extend to LATER tier** (issue #78) — KO/EC TSVs and Functional Annotation GFF require the same HTTP approach but are much larger (163+111 GB); streaming/chunked writes will be needed.